# Chapter 02：Triton 编程模型

**目标**：通过 copy kernel 理解 grid、program 和 block。重点不是性能，而是“哪个 program 负责哪些 offset”。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 1. Host 端观察 1D grid

grid 是 program 的数量。每个 program 使用自己的 id 计算起始位置。

In [ ]:
def demo_1d_grid(n_elements=20, block_size=8):
    grid = triton.cdiv(n_elements, block_size)
    print(f"grid=({grid},)")
    for program_id in range(grid):
        start = program_id * block_size
        stop = min(start + block_size, n_elements)
        print(f"program {program_id}: offsets [{start}, {stop})")

demo_1d_grid()

## 2. Copy kernel

`tl.arange(0, BLOCK_SIZE)` 生成一个向量。Triton program 对这个向量并行执行 load/store，而不是逐元素运行 Python 循环。

In [ ]:
@triton.jit
def copy_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    program_id = tl.program_id(axis=0)
    offsets = program_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    values = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, values, mask=mask)

def copy(x, block_size=256):
    if x.ndim != 1 or not x.is_cuda or not x.is_contiguous():
        raise ValueError("copy expects a contiguous 1D CUDA tensor")
    if block_size <= 0 or block_size & (block_size - 1):
        raise ValueError("block_size must be a positive power of two")
    y = torch.empty_like(x)
    if x.numel() > 0:
        copy_kernel[(triton.cdiv(x.numel(), block_size),)](
            x, y, x.numel(), BLOCK_SIZE=block_size
        )
    return y

## 3. 不同 BLOCK_SIZE

改变 block size 会改变 grid 大小，但不应改变结果。

In [ ]:
x = torch.randn(1003, device=device)
for block_size in (64, 128, 256):
    print(f"BLOCK_SIZE={block_size}, grid=({triton.cdiv(x.numel(), block_size)},)")
    assert_close(f"copy {block_size}", copy(x, block_size), x)

## 4. 轻量 2D grid 演示

二维 grid 中，一个 program 可用 `tl.program_id(0)` 和 `tl.program_id(1)` 得到两个坐标。这里先在 host 端打印所有坐标，不再引入新 kernel。

In [ ]:
def demo_2d_grid(rows=3, cols=10, block_cols=4):
    grid = (rows, triton.cdiv(cols, block_cols))
    print(f"2D grid={grid}")
    for row_program in range(grid[0]):
        for col_program in range(grid[1]):
            start = col_program * block_cols
            stop = min(start + block_cols, cols)
            print(f"program ({row_program}, {col_program}): row={row_program}, cols=[{start}, {stop})")

demo_2d_grid()

## 5. Benchmark

这里 benchmark 只验证完整章节结构；copy 性能会受输出分配和输入大小影响。

In [ ]:
x_bench = torch.randn(1_000_000, device=device)
print(f"PyTorch clone: {bench(lambda: x_bench.clone()):.3f} ms")
print(f"Triton copy:   {bench(lambda: copy(x_bench)):.3f} ms")

## 小结与练习

program 是 kernel 的一个实例，block 是该实例一次处理的向量范围，grid 是所有 program 的组织方式。

**练习**：用 `N=33, BLOCK_SIZE=16` 手工预测 grid 和每个 program 的 offset 范围，再运行 `demo_1d_grid(33, 16)`。